In [1]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, TypeAlias

import numpy as np
from numpy.typing import NDArray
from typeguard import typechecked


In [ ]:
FloatArray: TypeAlias = NDArray[np.float64]
TensorType: TypeAlias = tuple[int, int]
Index: TypeAlias = int | tuple[int, ...]
# PRUEBA GIT

In [4]:
def to_float_array(data: Any) -> FloatArray:
    return np.asarray(data, dtype=np.float64)


def validate_tensor_type(tensor_type: TensorType) -> None:
    r, s = tensor_type
    if r < 0 or s < 0:
        raise ValueError(f"Tipo tensorial inválido: {tensor_type}")


def validate_rank_matches_type(components: FloatArray, tensor_type: TensorType) -> None:
    expected_rank = tensor_type[0] + tensor_type[1]
    actual_rank = components.ndim
    if actual_rank != expected_rank:
        raise ValueError(
            f"Rank incompatible: tipo {tensor_type} exige rank {expected_rank}, "
            f"pero el array tiene rank {actual_rank}"
        )

In [5]:
@dataclass(frozen=True)
class Tensor:
    components: FloatArray
    tensor_type: TensorType

    def __post_init__(self) -> None:
        validate_tensor_type(self.tensor_type)
        validate_rank_matches_type(self.components, self.tensor_type)

    @property
    def rank(self) -> int:
        return self.components.ndim

    @property
    def shape(self) -> tuple[int, ...]:
        return self.components.shape

    def __getitem__(self, idx: Index):
        return self.components[idx]

    def __repr__(self) -> str:
        return (
            f"Tensor(tipo={self.tensor_type}, rank={self.rank}, shape={self.shape})\n"
            f"{self.components}"
        )

    @staticmethod
    def from_data(data: Any, tensor_type: TensorType) -> "Tensor":
        return Tensor(to_float_array(data), tensor_type)

In [6]:
v = Tensor.from_data([3, 4], (1, 0))
print(v)
print("rank =", v.rank)
print("shape =", v.shape)
print("v[0] =", v[0])

Tensor(tipo=(1, 0), rank=1, shape=(2,))
[3. 4.]
rank = 1
shape = (2,)
v[0] = 3.0


In [13]:
def vector(data: Any) -> Tensor:
    arr = to_float_array(data)
    if arr.ndim != 1:
        raise ValueError(f"Un vector debe ser 1D. shape recibido: {arr.shape}")
    return Tensor(arr, (1, 0))


def covector(data: Any) -> Tensor:
    arr = to_float_array(data)
    if arr.ndim != 1:
        raise ValueError(f"Un covector debe ser 1D. shape recibido: {arr.shape}")
    return Tensor(arr, (0, 1))


def endomorphism(data: Any) -> Tensor:
    arr = to_float_array(data)
    if arr.ndim != 2:
        raise ValueError(f"Un tensor (1,1) debe ser 2D. shape recibido: {arr.shape}")
    return Tensor(arr, (1, 1))


def bilinear_form(data: Any) -> Tensor:
    arr = to_float_array(data)
    if arr.ndim != 2:
        raise ValueError(f"Un tensor (0,2) debe ser 2D. shape recibido: {arr.shape}")
    return Tensor(arr, (0, 2))

In [14]:
v = vector([3, 4])
alpha = covector([2, -1])
A = endomorphism([[2, 1], [0, 3]])
g = bilinear_form([[2, 0], [0, 3]])

print(v)
print(alpha)
print(A)
print(g)

Tensor(tipo=(1, 0), rank=1, shape=(2,))
[3. 4.]
Tensor(tipo=(0, 1), rank=1, shape=(2,))
[ 2. -1.]
Tensor(tipo=(1, 1), rank=2, shape=(2, 2))
[[2. 1.]
 [0. 3.]]
Tensor(tipo=(0, 2), rank=2, shape=(2, 2))
[[2. 0.]
 [0. 3.]]


In [ ]:
np.transpose(alpha)+v

TypeError: unsupported operand type(s) for +: 'Tensor' and 'Tensor'

: 